# Notebook 03 — Exploratory Data Analysis
**Project:** Ola Bengaluru Rides — Cancellation Intelligence & Revenue Recovery  
**Dataset:** Bengaluru Ola Rides, January 2024  
**Author:** Joshit  
**Institute:** Newton School of Technology — DVA Capstone 2

Using the cleaned dataset from Notebook 02. Goal here is to understand patterns in cancellations, revenue loss, peak hours and area-wise behaviour.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

print('ready.')

In [ ]:
# update path if running locally: '../data/processed/ola_bengaluru_cleaned.csv'
CLEAN_PATH = '/content/ola_bengaluru_cleaned.csv'

df = pd.read_csv(CLEAN_PATH)
df['date'] = pd.to_datetime(df['date'])
print(f'loaded: {df.shape[0]:,} rows, {df.shape[1]} columns')

### Quick overview of the cleaned data

In [ ]:
total = len(df)
successful = df['is_successful'].sum()
cancelled = df['is_cancelled'].sum()
incomplete = (df['booking_status'] == 'Incomplete').sum()
total_revenue = df['booking_value'].sum()
total_lost = df['revenue_lost'].sum()

print(f'total rides        : {total:,}')
print(f'successful         : {successful:,} ({successful/total*100:.1f}%)')
print(f'cancelled          : {cancelled:,} ({cancelled/total*100:.1f}%)')
print(f'incomplete         : {incomplete:,} ({incomplete/total*100:.1f}%)')
print(f'total revenue      : ₹{total_revenue:,.0f}')
print(f'estimated rev lost : ₹{total_lost:,.0f}')

### Cancellation rate by vehicle type
Wanted to see if certain vehicle types are more prone to cancellations — this directly ties to our problem statement.

In [ ]:
vehicle_stats = df.groupby('vehicle_type').agg(
    total=('booking_id', 'count'),
    cancelled=('is_cancelled', 'sum'),
    successful=('is_successful', 'sum')
).reset_index()

vehicle_stats['cancel_rate'] = (vehicle_stats['cancelled'] / vehicle_stats['total'] * 100).round(2)
vehicle_stats = vehicle_stats.sort_values('cancel_rate', ascending=False)
print(vehicle_stats.to_string(index=False))

In [ ]:
plt.figure(figsize=(10, 5))
colors = ['#E74C3C' if r > vehicle_stats['cancel_rate'].mean() else '#3498DB'
          for r in vehicle_stats['cancel_rate']]
bars = plt.bar(vehicle_stats['vehicle_type'], vehicle_stats['cancel_rate'],
               color=colors, edgecolor='white')
plt.axhline(vehicle_stats['cancel_rate'].mean(), color='black', linestyle='--',
            linewidth=1.2, label=f'avg: {vehicle_stats["cancel_rate"].mean():.1f}%')
plt.title('Cancellation Rate by Vehicle Type', fontsize=13, fontweight='bold')
plt.ylabel('Cancellation Rate (%)')
plt.xlabel('Vehicle Type')
plt.legend()
for bar, val in zip(bars, vehicle_stats['cancel_rate']):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
             f'{val}%', ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('/content/03_cancel_by_vehicle.png', dpi=150, bbox_inches='tight')
plt.show()

### Cancellation reasons — driver vs customer
Breaking down what's actually causing cancellations on both sides.

In [ ]:
driver_reasons = df[df['cancellation_party'] == 'Driver']['cancel_reason_driver'].value_counts()
driver_reasons = driver_reasons[driver_reasons.index != 'Not Applicable']

customer_reasons = df[df['cancellation_party'] == 'Customer']['cancel_reason_customer'].value_counts()
customer_reasons = customer_reasons[customer_reasons.index != 'Not Applicable']

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].barh(driver_reasons.index, driver_reasons.values, color='#E74C3C', edgecolor='white')
axes[0].set_title('Driver Cancellation Reasons', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Count')
for i, val in enumerate(driver_reasons.values):
    axes[0].text(val + 30, i, f'{val:,}', va='center', fontsize=9)

axes[1].barh(customer_reasons.index, customer_reasons.values, color='#E67E22', edgecolor='white')
axes[1].set_title('Customer Cancellation Reasons', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Count')
for i, val in enumerate(customer_reasons.values):
    axes[1].text(val + 10, i, f'{val:,}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('/content/03_cancel_reasons.png', dpi=150, bbox_inches='tight')
plt.show()

### Peak hour analysis
Checking which hours of the day have the highest cancellation rates — useful for operational decisions around driver allocation.

In [ ]:
hourly = df.groupby('hour').agg(
    total=('booking_id', 'count'),
    cancelled=('is_cancelled', 'sum'),
    successful=('is_successful', 'sum')
).reset_index()

hourly['cancel_rate'] = (hourly['cancelled'] / hourly['total'] * 100).round(2)

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

axes[0].bar(hourly['hour'], hourly['total'], color='#3498DB', edgecolor='white', label='Total Rides')
axes[0].bar(hourly['hour'], hourly['cancelled'], color='#E74C3C', edgecolor='white', label='Cancelled')
axes[0].set_title('Ride Volume and Cancellations by Hour', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Number of Rides')
axes[0].set_xticks(range(0, 24))
axes[0].legend()

axes[1].plot(hourly['hour'], hourly['cancel_rate'], color='#E74C3C', linewidth=2, marker='o', markersize=4)
axes[1].fill_between(hourly['hour'], hourly['cancel_rate'], alpha=0.15, color='#E74C3C')
axes[1].axhline(hourly['cancel_rate'].mean(), color='black', linestyle='--',
                linewidth=1, label=f'avg: {hourly["cancel_rate"].mean():.1f}%')
axes[1].set_title('Cancellation Rate by Hour of Day', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Cancellation Rate (%)')
axes[1].set_xlabel('Hour of Day')
axes[1].set_xticks(range(0, 24))
axes[1].legend()

plt.tight_layout()
plt.savefig('/content/03_peak_hours.png', dpi=150, bbox_inches='tight')
plt.show()

### Cancellations by time of day bucket

In [ ]:
order = ['Early Morning', 'Morning', 'Afternoon', 'Evening', 'Night', 'Late Night']
tod = df.groupby('time_of_day').agg(
    total=('booking_id', 'count'),
    cancelled=('is_cancelled', 'sum')
).reindex(order).reset_index()
tod['cancel_rate'] = (tod['cancelled'] / tod['total'] * 100).round(2)

plt.figure(figsize=(10, 5))
bars = plt.bar(tod['time_of_day'], tod['cancel_rate'], color='#9B59B6', edgecolor='white')
plt.title('Cancellation Rate by Time of Day', fontsize=13, fontweight='bold')
plt.ylabel('Cancellation Rate (%)')
plt.xlabel('Time of Day')
for bar, val in zip(bars, tod['cancel_rate']):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
             f'{val}%', ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('/content/03_time_of_day.png', dpi=150, bbox_inches='tight')
plt.show()

### Day of week patterns
Checking if weekdays vs weekends show any difference in cancellation behaviour.

In [ ]:
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
dow = df.groupby('day_of_week').agg(
    total=('booking_id', 'count'),
    cancelled=('is_cancelled', 'sum'),
    revenue=('booking_value', 'sum')
).reindex(day_order).reset_index()
dow['cancel_rate'] = (dow['cancelled'] / dow['total'] * 100).round(2)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(dow['day_of_week'], dow['total'], color='#3498DB', edgecolor='white')
axes[0].set_title('Total Rides by Day of Week', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Rides')
axes[0].tick_params(axis='x', rotation=30)

axes[1].plot(dow['day_of_week'], dow['cancel_rate'], color='#E74C3C',
             linewidth=2, marker='o', markersize=6)
axes[1].set_title('Cancellation Rate by Day of Week', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Cancellation Rate (%)')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('/content/03_day_of_week.png', dpi=150, bbox_inches='tight')
plt.show()

### Area-wise cancellation analysis
Finding which pickup zones have the highest cancellation rates — key for fleet reallocation recommendations.

In [ ]:
area_stats = df.groupby('pickup_location').agg(
    total=('booking_id', 'count'),
    cancelled=('is_cancelled', 'sum'),
    revenue_lost=('revenue_lost', 'sum')
).reset_index()

area_stats['cancel_rate'] = (area_stats['cancelled'] / area_stats['total'] * 100).round(2)
area_stats['area_num'] = area_stats['pickup_location'].str.extract(r'(\d+)').astype(int)
area_stats = area_stats.sort_values('cancel_rate', ascending=False)

print('top 10 areas by cancellation rate:')
print(area_stats[['pickup_location', 'total', 'cancelled', 'cancel_rate', 'revenue_lost']].head(10).to_string(index=False))

In [ ]:
top10 = area_stats.head(10)
bottom10 = area_stats.tail(10)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].barh(top10['pickup_location'], top10['cancel_rate'], color='#E74C3C', edgecolor='white')
axes[0].set_title('Top 10 Areas — Highest Cancellation Rate', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Cancellation Rate (%)')
for i, val in enumerate(top10['cancel_rate']):
    axes[0].text(val + 0.1, i, f'{val}%', va='center', fontsize=9)

axes[1].barh(bottom10['pickup_location'], bottom10['cancel_rate'], color='#2ECC71', edgecolor='white')
axes[1].set_title('Top 10 Areas — Lowest Cancellation Rate', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Cancellation Rate (%)')
for i, val in enumerate(bottom10['cancel_rate']):
    axes[1].text(val + 0.1, i, f'{val}%', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('/content/03_area_cancellation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# heatmap: vehicle type vs area (top 15 areas by volume)
top_areas = df['pickup_location'].value_counts().head(15).index
heatmap_data = df[df['pickup_location'].isin(top_areas)].groupby(
    ['pickup_location', 'vehicle_type'])['is_cancelled'].mean().unstack() * 100

plt.figure(figsize=(13, 7))
sns.heatmap(heatmap_data.round(1), annot=True, fmt='.1f', cmap='RdYlGn_r',
            linewidths=0.5, cbar_kws={'label': 'Cancellation Rate (%)'})
plt.title('Cancellation Rate % — Area vs Vehicle Type (Top 15 Areas)', fontsize=13, fontweight='bold')
plt.xlabel('Vehicle Type')
plt.ylabel('Pickup Area')
plt.tight_layout()
plt.savefig('/content/03_area_vehicle_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

### Revenue analysis
Looking at revenue by vehicle type and how much is being lost to cancellations.

In [ ]:
rev_stats = df.groupby('vehicle_type').agg(
    total_revenue=('booking_value', 'sum'),
    avg_fare=('booking_value', 'mean'),
    revenue_lost=('revenue_lost', 'sum'),
    successful=('is_successful', 'sum')
).reset_index().sort_values('total_revenue', ascending=False)

print(rev_stats.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(rev_stats['vehicle_type'], rev_stats['total_revenue'] / 1e6,
            color='#2ECC71', edgecolor='white')
axes[0].set_title('Total Revenue by Vehicle Type (₹ Million)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Revenue (₹ Million)')
axes[0].tick_params(axis='x', rotation=30)

axes[1].bar(rev_stats['vehicle_type'], rev_stats['revenue_lost'] / 1e6,
            color='#E74C3C', edgecolor='white')
axes[1].set_title('Estimated Revenue Lost by Vehicle Type (₹ Million)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Revenue Lost (₹ Million)')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('/content/03_revenue_by_vehicle.png', dpi=150, bbox_inches='tight')
plt.show()

### Payment method distribution

In [ ]:
payment = df[df['is_successful'] == 1]['payment_method'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
colors = ['#3498DB', '#2ECC71', '#E67E22', '#9B59B6']

axes[0].pie(payment.values, labels=payment.index, colors=colors,
            autopct='%1.1f%%', startangle=140, textprops={'fontsize': 10})
axes[0].set_title('Payment Method Split', fontsize=12, fontweight='bold')

pay_rev = df[df['is_successful'] == 1].groupby('payment_method')['booking_value'].mean().sort_values(ascending=False)
axes[1].bar(pay_rev.index, pay_rev.values, color=colors, edgecolor='white')
axes[1].set_title('Avg Fare by Payment Method', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Avg Booking Value (₹)')
for i, val in enumerate(pay_rev.values):
    axes[1].text(i, val + 5, f'₹{val:.0f}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('/content/03_payment_method.png', dpi=150, bbox_inches='tight')
plt.show()

### Ratings distribution
Checking driver and customer rating distributions — and whether lower ratings correlate with higher cancellation areas.

In [ ]:
df_s = df[df['is_successful'] == 1]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].hist(df_s['driver_rating'].dropna(), bins=20, color='#3498DB',
             edgecolor='white', rwidth=0.85)
axes[0].set_title('Driver Rating Distribution', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Rating')
axes[0].set_ylabel('Count')
axes[0].axvline(df_s['driver_rating'].mean(), color='red', linestyle='--',
                label=f'mean: {df_s["driver_rating"].mean():.2f}')
axes[0].legend()

axes[1].hist(df_s['customer_rating'].dropna(), bins=20, color='#2ECC71',
             edgecolor='white', rwidth=0.85)
axes[1].set_title('Customer Rating Distribution', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Rating')
axes[1].set_ylabel('Count')
axes[1].axvline(df_s['customer_rating'].mean(), color='red', linestyle='--',
                label=f'mean: {df_s["customer_rating"].mean():.2f}')
axes[1].legend()

plt.tight_layout()
plt.savefig('/content/03_ratings_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

### VTAT and CTAT analysis
Checking average vehicle arrival times and customer arrival times by vehicle type — high VTAT likely contributes to cancellations.

In [ ]:
wait_times = df[df['is_successful'] == 1].groupby('vehicle_type').agg(
    avg_vtat=('avg_vtat', 'mean'),
    avg_ctat=('avg_ctat', 'mean')
).reset_index().round(2)

x = range(len(wait_times))
width = 0.35

plt.figure(figsize=(11, 5))
plt.bar([i - width/2 for i in x], wait_times['avg_vtat'], width, label='Avg VTAT', color='#3498DB', edgecolor='white')
plt.bar([i + width/2 for i in x], wait_times['avg_ctat'], width, label='Avg CTAT', color='#E67E22', edgecolor='white')
plt.xticks(x, wait_times['vehicle_type'])
plt.title('Avg VTAT vs CTAT by Vehicle Type (Successful Rides)', fontsize=13, fontweight='bold')
plt.ylabel('Minutes')
plt.legend()
plt.tight_layout()
plt.savefig('/content/03_vtat_ctat.png', dpi=150, bbox_inches='tight')
plt.show()

### Key insights from EDA

These are the patterns that stood out — written in business language for the report:

1. **33% of rides never complete** — driver cancellations alone account for 19.2%, nearly 3x customer cancellations (7.6%)
2. **Driver cancellation reasons are operational** — 'More than permitted people' and 'Personal & Car issues' are the top two, both preventable through better driver screening and vehicle checks
3. **Customer cancellations are mostly Ola's fault** — 'Driver not moving towards pickup' is the top reason, suggesting GPS or driver compliance issues
4. **Certain areas consistently underperform** — the top 10 high-cancellation zones are identifiable and targeted intervention is possible
5. **Late night and early morning hours show elevated cancellation rates** — driver availability during off-peak hours is a problem
6. **Revenue lost per month is significant** — non-successful rides at avg ₹1,023 per ride represent a recoverable opportunity
7. **Vehicle types show different cancellation profiles** — premium vehicles (Prime SUV, Prime Plus) tend to have higher driver cancellation rates
8. **Payment methods are evenly distributed** — no single method dominates, UPI and Cash are slightly higher

In [ ]:
# saving EDA summary
eda_summary = df.groupby('vehicle_type').agg(
    total_rides=('booking_id', 'count'),
    successful=('is_successful', 'sum'),
    cancelled=('is_cancelled', 'sum'),
    cancel_rate=('is_cancelled', 'mean'),
    avg_fare=('booking_value', 'mean'),
    revenue_lost=('revenue_lost', 'sum')
).round(2).reset_index()

eda_summary['cancel_rate'] = (eda_summary['cancel_rate'] * 100).round(2)
eda_summary.to_csv('/content/03_eda_summary.csv', index=False)
print('saved.')
print('proceed to 04_statistical_analysis.ipynb')